# Analyse Exploratoire des Données (EDA) — Titanic
**Objectif :** Comprendre ce que contient un dataset, détecter les problèmes de qualité, et le préparer pour l'analyse.  
On n'interprète pas encore — on **observe**, on **questionne**, on **décide**.

## Qu'est-ce que l'EDA ?

L'analyse exploratoire des données, ou *Exploratory Data Analysis* (EDA) en anglais, est la phase préliminaire de tout projet data.

L'EDA répond à trois questions dans l'ordre :
- **Qu'est-ce que j'ai ?** — structure, types, dimensions, premières statistiques
- **Est-ce que c'est propre ?** — manquants, doublons, outliers, incohérences
- **Qu'est-ce que ça dit ?** — distributions, relations entre variables, premières hypothèses

### Pourquoi c'est indispensable

Voir l'exemple du quartet d'Anscombe : quatre datasets avec les mêmes statistiques descriptives (même moyenne, même variance, même corrélation) mais des structures radicalement différentes. Sans visualisation, on les traite de la même façon — et on se trompe sur trois des quatre.

Le Datasaurus Dozen pousse l'idée plus loin : douze datasets avec des statistiques identiques, dont un en forme de dinosaure.

**Règle fondamentale : explorer avant de modéliser.** Un modèle entraîné sur des données non explorées hérite de tous leurs problèmes sans qu'on le sache.

### Ce que l'EDA n'est pas

L'EDA n'est pas une étape à cocher mécaniquement. Ce n'est pas non plus de l'analyse — on ne tire pas encore de conclusions sur les relations causales entre variables. C'est une phase d'**observation structurée** dont l'unique objectif est de comprendre ce avec quoi on travaille avant d'agir.



## Sommaire

**Partie 1 — Exploration et nettoyage**
- 1.1 Inspection initiale
- 1.2 Renommage et types
- 1.3 Doublons
- 1.4 Valeurs manquantes
- 1.5 Sélection des colonnes
- 1.6 Outliers
- 1.7 Distributions
- 1.8 Nettoyage

<u>**Imports et chargement**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

#Partie 1 : Exploration et Nettoyage

## 1.1 Inspection initiale

Premier contact avec le dataset : dimensions, types, aperçu des valeurs, statistiques descriptives.  
**On ne modifie rien à cette étape.**

In [ ]:
# Dimensions : nombre de lignes et de colonnes
print(df.shape)

In [ ]:
# Types de données et valeurs non nulles par colonne
df.info()

In [ ]:
# Aperçu des premières lignes
df.head()

In [ ]:
# Statistiques descriptives : mean, std, min, max, quartiles
df.describe(include='all')

## 1.2. Renommage des colonnes et types de données

### 1.2.1 Renommage
Standardiser les noms de colonnes dès le départ : snake_case, minuscules, pas d'espaces.  
Travailler avec des noms propres évite les erreurs silencieuses dans la suite du code.

- `str.strip()` — supprime les espaces en début et fin de nom
- `str.lower()` — convertit en minuscules  
- `str.replace(' ', '_')` — remplace les espaces internes par un underscore

In [ ]:
# Standardisation globale des noms de colonnes
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
# Renommage ciblé de colonnes spécifiques
# Syntaxe : {'ancien_nom': 'nouveau_nom'}
df.rename(columns={'passengerid': 'id', 'pclass': 'p_class'}, inplace=True)

df.columns

### 1.2.2 Types de données

Le type Pandas ne reflète pas toujours la nature réelle d'une variable.  
`survived` et `p_class` sont stockés en `int64` mais représentent des catégories — pas des quantités sur lesquelles faire des calculs.

**`object` vs `category` :**
- `object` — stocke chaque valeur indépendamment en mémoire (liste de strings Python)
- `category` — stocke les valeurs uniques une seule fois et remplace chaque occurrence par un index entier. Plus économique en mémoire sur les colonnes à faible cardinalité.

`category` accepte des chiffres ou des mots — ce qui compte c'est la faible cardinalité, pas la nature des valeurs.

Candidats `category` sur le Titanic :
- `sex` — catégories textuelles (`male`/`female`), actuellement en `object`
- `p_class` — entiers représentant des rangs (`1`, `2`, `3`), pas des quantités
- `survived` — binaire (`0`/`1`), sémantiquement un booléen

In [ ]:
df[['survived', 'p_class', 'sex']] = df[['survived', 'p_class', 'sex']].astype('category')

df.info()

## 1.3. Détection et suppression des doublons

Action simple et sans ambiguïté — à faire tôt car les doublons faussent toutes les statistiques suivantes.

In [ ]:
print(f"Doublons détectés : {df.duplicated().sum()}")

In [ ]:
# Si doublons détectés : supprimer en gardant la première occurrence
df.drop_duplicates(inplace=True)
# Pour garder la dernière occurrence : df.drop_duplicates(keep='last', inplace=True)

## 1.4. Valeurs manquantes — observation

**À cette étape on observe uniquement. On ne traite pas encore.**  
L'objectif est de comprendre le mécanisme du manquant avant de choisir une stratégie.

### Les trois types de manquants

**MCAR — Missing Completely At Random**  
Le manquant n'a aucun lien avec aucune variable. Hasard pur. Imputer ou supprimer n'introduit pas de biais.

**MAR — Missing At Random**  
Le manquant est lié à d'autres variables *observées*, mais pas à la valeur manquante elle-même.  
Exemple : les passagers de 3ème classe ont plus de valeurs manquantes sur `age` parce que leurs données étaient moins bien collectées — pas parce qu'ils sont plus jeunes ou plus vieux.  
Stratégie : imputer par groupe.

**MNAR — Missing Not At Random**  
Le manquant est lié à la valeur manquante elle-même.  
Exemple : `cabin` manque surtout en 3ème classe parce que ces passagers n'avaient pas de cabine assignée — l'absence *est* l'information.  
Cas le plus problématique : aucune imputation ne corrige vraiment le biais.

In [ ]:
# Quantification des manquants par colonne
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'n': missing, '%': missing_pct})[missing > 0])

In [ ]:
# Carte visuelle des manquants
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Carte des valeurs manquantes')
plt.show()

### Investiguer le mécanisme du manquant

Deux méthodes complémentaires pour tester si le manquant est structuré ou aléatoire.

In [ ]:
# Méthode 1 — Taux de manquants par groupe
# Si le taux varie selon p_class, le manquant est structuré (MAR au minimum)
print("Taux de manquants sur 'age' par classe :")
print(df.groupby('p_class', observed=True)['age'].apply(lambda x: x.isnull().mean()).round(2))

print("\nTaux de manquants sur 'cabin' par classe :")
print(df.groupby('p_class', observed=True)['cabin'].apply(lambda x: x.isnull().mean()).round(2))

**Lecture des résultats :**
- `cabin` : 19% de manquants en 1ère classe, 91% en 2ème, 98% en 3ème → lien fort avec `p_class` → **MAR** (ou MNAR si les passagers de 3ème classe n'avaient simplement pas de cabine assignée)
- `age` : variation modérée (6% à 28%) → lien faible → **MAR faible**, pas MCAR

In [ ]:
# Méthode 2 — Corrélation entre indicatrice de manquant et variables numériques
# Plus rigoureux : teste le lien avec toutes les variables numériques en une fois
df['age_missing'] = df['age'].isnull().astype(int)
df[['age_missing', 'fare', 'sibsp', 'parch']].corr()

**Lecture :** `age_missing` corrélé à -0.10 avec `fare` — les passagers qui paient plus ont plus souvent leur âge renseigné.  
Signal réel mais faible (< 0.13). Confirme MAR faible.  
⚠️ Cette méthode ne détecte que les liens linéaires.

## 1.5. Sélection des colonnes

Après l'inspection et l'analyse des manquants, on dispose de toutes les informations pour décider quelles colonnes conserver.

**Décisions sur le Titanic :**
- `name`, `ticket` → supprimées (pas pertinentes pour l'analyse)
- `cabin` → remplacée par une indicatrice `has_cabin` (77% de manquants, mais l'absence est informative)
- `id` → conservé (clé potentielle si jointure avec d'autres fichiers)
- `sibsp`, `parch` → conservés (variables familiales potentiellement liées à la survie)
- `fare` → conservé malgré l'asymétrie (granularité continue que `p_class` n'a pas)

**`.copy()` est indispensable** — sans lui, `df_clean` est une vue de `df` et toute modification déclencherait un `SettingWithCopyWarning`.

In [ ]:
df_clean = df[['id', 'survived', 'p_class', 'sex', 'age', 'sibsp', 'parch', 'fare', 'cabin', 'embarked']].copy()

df_clean.shape

## 1.6. Outliers

Un outlier est une valeur qui s'éloigne significativement du reste de la distribution.  
La bonne question n'est pas *"est-ce une erreur ?"* mais *"est-ce que cette valeur est plausible dans le contexte ?"*

Trois cas possibles :
- **Erreur** — âge de 999 ans, tarif négatif → corriger ou supprimer
- **Valeur rare mais réelle** — un billet à 512£ sur le Titanic → ne pas toucher
- **Information importante** — en détection de fraude, l'outlier *est* ce qu'on cherche

In [ ]:
# Visualisation des outliers sur fare
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df_clean['fare'], ax=axes[0])
axes[0].set_title('fare — boxplot')
sns.histplot(df_clean['fare'], bins=50, ax=axes[1])
axes[1].set_title('fare — distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Investiguer les billets à 0£
print(f"Billets à 0£ : {(df_clean['fare'] == 0).sum()}")
df_clean[df_clean['fare'] == 0]

**Observations sur les 15 billets à 0£ :**
- Tous des hommes, tous embarqués à Southampton (S)
- `survived = 0` pour tous — aucun n'a survécu
- `sibsp` et `parch` à 0 — voyagent seuls
- Beaucoup d'âges manquants

**Hypothèse :** membres d'équipage ou personnel embarqués comme passagers.  
→ Ne pas supprimer, ne pas imputer `fare`. Créer une indicatrice `is_crew` pour les isoler.

In [ ]:
# Investiguer les billets > 100£ (53 passagers)
df_clean[df_clean['fare'] > 100]

**Observations sur les billets > 100£ :**
- Quasi exclusivement en 1ère classe
- Majorité de femmes — taux de survie très élevé
- `cabin` renseignée pour la majorité → confirme le MAR sur `cabin`
- Les 2 billets à 512£ ont des profils cohérents

**Conclusion :** les valeurs élevées sont réelles. Pas de suppression, pas d'imputation.

## 1.7. Distributions des variables numériques

Étape de lecture — on visualise la forme des distributions avant d'agir.  
Les NaN sont ignorés par défaut par `.hist()`.

In [ ]:
df_clean[['age', 'fare', 'sibsp', 'parch']].hist(figsize=(10, 6), bins=20)
plt.tight_layout()
plt.show()

**Observations :**
- `age` — distribution asymétrique à droite, pic entre 20-35 ans. Pic à gauche (< 5 ans) réel : familles d'émigrants en 3ème classe.
- `fare` — très fortement asymétrique, écrasante majorité sous 100£, quelques valeurs jusqu'à 512£. Cohérent avec l'analyse outliers.
- `sibsp` et `parch` — majorité voyage seule (valeur 0 dominante). Distribution attendue.

In [ ]:
# Vérification du pic enfants < 5 ans
count = (df_clean['age'] < 5).sum()
pct = round(count / df_clean['age'].notna().sum() * 100, 1)
print(f"Enfants < 5 ans : {count} ({pct}% des âges renseignés)")
print(f"Âges manquants : {df_clean['age'].isnull().sum()}")

## 1.8. Nettoyage

Toutes les décisions sont prises. On agit maintenant.

**Récapitulatif des décisions :**
- `cabin` → indicatrice `has_cabin` (1 si cabine renseignée, 0 sinon) puis suppression
- `fare == 0` → indicatrice `is_crew` pour isoler ces passagers
- `age` → imputation par médiane **par groupe `p_class`** (MAR faible, distribution asymétrique)
- `embarked` manquants → a supprimer
- Doublons → déjà vérifiés à l'étape 3

⚠️ **Ordre important :** les suppressions de lignes (doublons, `embarked`) doivent précéder l'imputation de `age` — sinon les médianes par groupe sont calculées sur un dataset légèrement différent.

In [ ]:
df_clean.dropna(subset=['embarked'], inplace=True)
print(f"Lignes restantes : {df_clean.shape[0]}")

In [ ]:
# cabin → indicatrice has_cabin
# notna() renvoie True/False, astype(int) convertit en 1/0
df_clean['has_cabin'] = df_clean['cabin'].notna().astype(int)
df_clean = df_clean.drop('cabin', axis=1)

In [ ]:
# fare == 0 → indicatrice is_crew_?
df_clean['is_crew_?'] = (df_clean['fare'] == 0).astype(int)

In [ ]:
# Vérification des médianes par groupe avant imputation
print("Médianes d'âge par classe :")
print(df_clean.groupby('p_class', observed=True)['age'].median())

In [ ]:
# age → imputation par médiane par groupe p_class
# transform() applique la fonction par groupe et renvoie le résultat
# dans la même forme que le dataframe original
df_clean['age'] = df_clean.groupby('p_class', observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}")

In [ ]:
# Vérification finale
df_clean.info()

In [ ]:
df_clean.head()